# Notebook 12 — ROPE & Decision Theory

## Background

Getting a posterior is half the job. The other half is turning it into a decision. In the frequentist tradition, decisions are made by comparing p-values to thresholds: p < 0.05 means 'reject H0'. This is elegant but fragile — it collapses a continuous quantity (the posterior) into a binary accept/reject, and it doesn't encode anything about how *much* the effect matters.

Decision theory provides a principled framework: to make a decision, you need a **loss function** (what does each possible decision cost you in each possible state of the world?) and a **posterior** (what states of the world are plausible?). The optimal decision minimizes expected posterior loss.

The **Region of Practical Equivalence (ROPE)** is a decision-theoretic idea from Kruschke (2011): define a range of values `[-delta, +delta]` around zero (or some null value) within which the effect is practically indistinguishable from zero. Then ask: does the posterior fall inside the ROPE (treat as equivalent to null), outside the ROPE (treat as meaningful), or straddle it (inconclusive)?

## What You'll Learn

- ROPE: defining practical equivalence and making decisions from posteriors
- The three-way decision rule: inside ROPE / outside ROPE / inconclusive
- Loss functions: asymmetric costs change optimal decisions
- Expected posterior loss minimization
- ROPE on A/B test posteriors: when is a conversion rate difference meaningful?
- Highest Density Interval (HDI) vs credible interval

## Why This Matters

ROPE answers a question that p-values can't: is the effect *practically important*? A statistically significant effect (p < 0.001) with a posterior entirely within the ROPE is not worth acting on. A marginally significant effect (p = 0.06) with a posterior entirely outside the ROPE should absolutely inform your decision. Decision theory unifies these considerations into a principled framework.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — HDI vs Credible Interval

Two common ways to summarize a posterior:

**Equal-tailed credible interval (ETI)**: cut equal probability from each tail. The 95% ETI is `[2.5th percentile, 97.5th percentile]`. Analogous to the frequentist confidence interval.

**Highest Density Interval (HDI)**: the shortest interval containing X% of the probability mass. For symmetric unimodal distributions, ETI and HDI agree. For skewed or multimodal distributions, HDI is more informative — it captures the region of highest posterior density.

HDI is what you want for ROPE analysis: it tells you the most credible range of values.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

def hdi(samples, prob=0.94):
    """Highest Density Interval via sorted-window method."""
    sorted_samp = np.sort(samples)
    n = len(sorted_samp)
    width = int(np.floor(prob * n))
    # Find the shortest window of 'width' samples
    window_widths = sorted_samp[width:] - sorted_samp[:n - width]
    min_idx = np.argmin(window_widths)
    return sorted_samp[min_idx], sorted_samp[min_idx + width]

# Compare ETI vs HDI on different posterior shapes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

distributions = [
    ('Symmetric Normal\n(ETI = HDI)', np.random.normal(0, 1, 10000)),
    ('Right-Skewed (Beta(2,8))\n(HDI shifts left)', np.random.beta(2, 8, 10000)),
    ('Bimodal\n(HDI finds dense region)', np.concatenate([
        np.random.normal(-2, 0.5, 5000),
        np.random.normal(2, 0.5, 5000)
    ])),
]

for ax, (label, samples) in zip(axes, distributions):
    # ETI
    eti_lo, eti_hi = np.percentile(samples, [3, 97])
    # HDI
    hdi_lo, hdi_hi = hdi(samples, prob=0.94)
    
    xr = np.linspace(samples.min(), samples.max(), 300)
    kde = stats.gaussian_kde(samples)
    ax.fill_between(xr, kde(xr), alpha=0.3, color='gray')
    ax.plot(xr, kde(xr), 'gray', lw=1.5)
    
    # ETI
    ax.axvspan(eti_lo, eti_hi, alpha=0.25, color='darkorange',
               label=f'94% ETI: [{eti_lo:.2f}, {eti_hi:.2f}]')
    # HDI
    ax.axvspan(hdi_lo, hdi_hi, alpha=0.25, color='steelblue',
               label=f'94% HDI: [{hdi_lo:.2f}, {hdi_hi:.2f}]')
    
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('theta')

plt.suptitle('Equal-Tailed Interval (ETI) vs Highest Density Interval (HDI)', fontsize=12)
plt.tight_layout()
plt.show()

print('HDI: shortest interval containing 94% of posterior mass')
print('ETI: equal probability cut from each tail')
print('Difference is largest for skewed/bimodal posteriors -- use HDI for ROPE analysis')

## Part 2 — ROPE: Region of Practical Equivalence

The ROPE operationalizes the question: 'Is this effect big enough to matter?'

1. **Define the ROPE**: `[-delta, +delta]` around zero (or any null value). `delta` is domain-specific — it's the smallest effect you'd care about.
   - For a drug trial: ROPE might be ±0.1 SD (effects smaller than this are clinically negligible)
   - For an A/B test on a 0.05% conversion rate: ROPE might be ±0.001
   - For a revenue impact: ROPE might be ±$1/user

2. **Compare the HDI to the ROPE**:
   - **HDI entirely inside ROPE**: accept H0 (effect is practically null) — can declare equivalence
   - **HDI entirely outside ROPE**: reject H0 (effect is practically meaningful)
   - **HDI overlaps ROPE**: inconclusive — need more data

3. **Act on the decision**: ROPE turns posterior uncertainty into a three-way decision with explicit context about what 'matters'.

In [ ]:
def rope_decision(samples, rope_lo, rope_hi, hdi_prob=0.94):
    """Evaluate ROPE decision for posterior samples.
    
    Returns:
    - hdi: (lo, hi) of the HDI
    - p_inside_rope: fraction of posterior inside ROPE
    - decision: 'accept_null' | 'reject_null' | 'inconclusive'
    """
    hdi_lo, hdi_hi = hdi(samples, prob=hdi_prob)
    p_inside = np.mean((samples >= rope_lo) & (samples <= rope_hi))
    
    hdi_inside_rope  = (hdi_lo >= rope_lo) and (hdi_hi <= rope_hi)
    hdi_outside_rope = (hdi_hi <= rope_lo) or (hdi_lo >= rope_hi)
    
    if hdi_inside_rope:
        decision = 'accept_null (equivalent to zero)'
    elif hdi_outside_rope:
        decision = 'reject_null (practically meaningful)'
    else:
        decision = 'inconclusive (overlap with ROPE)'
    
    return {
        'hdi': (hdi_lo, hdi_hi),
        'p_inside_rope': p_inside,
        'decision': decision,
    }

# Demonstrate ROPE on different posterior scenarios
rope_lo, rope_hi = -0.05, 0.05  # effect < 5% is practically negligible

scenarios = [
    ('Large effect\n(clearly meaningful)', np.random.normal(0.25, 0.04, 10000)),
    ('Small effect\n(practically null)', np.random.normal(0.01, 0.015, 10000)),
    ('Uncertain\n(inconclusive)', np.random.normal(0.06, 0.05, 10000)),
    ('Negative effect\n(clearly harmful)', np.random.normal(-0.15, 0.03, 10000)),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (label, samples) in zip(axes, scenarios):
    result = rope_decision(samples, rope_lo, rope_hi)
    hdi_lo_r, hdi_hi_r = result['hdi']
    
    xr = np.linspace(samples.min() - 0.05, samples.max() + 0.05, 300)
    kde = stats.gaussian_kde(samples)
    ax.fill_between(xr, kde(xr), alpha=0.4, color='steelblue')
    ax.plot(xr, kde(xr), 'steelblue', lw=1.5)
    
    # ROPE
    ax.axvspan(rope_lo, rope_hi, alpha=0.15, color='gray', label='ROPE')
    ax.axvline(rope_lo, color='gray', linestyle='--', lw=1)
    ax.axvline(rope_hi, color='gray', linestyle='--', lw=1)
    
    # HDI
    ax.axvline(hdi_lo_r, color='red', lw=2, linestyle='-')
    ax.axvline(hdi_hi_r, color='red', lw=2, linestyle='-', label=f'94% HDI')
    ax.axhline(0, color='red', lw=0.5)  # dummy for legend
    
    decision = result['decision']
    color = 'green' if 'reject' in decision else ('red' if 'accept' in decision else 'darkorange')
    ax.set_title(f'{label}\n{decision}', fontsize=8, color=color)
    ax.set_xlabel('Effect size')
    ax.legend(fontsize=6)

plt.suptitle(f'ROPE Decisions (ROPE = [{rope_lo}, {rope_hi}])\n'
             f'Gray = ROPE; Red lines = 94% HDI', fontsize=12)
plt.tight_layout()
plt.show()

for label, samples in scenarios:
    result = rope_decision(samples, rope_lo, rope_hi)
    print(f'{label.replace(chr(10)," "):35} HDI=[{result["hdi"][0]:.3f},{result["hdi"][1]:.3f}]  '
          f'P(inside ROPE)={result["p_inside_rope"]:.2f}  {result["decision"]}')

## Part 3 — Loss Functions and Expected Posterior Loss

ROPE is a special case of decision theory. The general framework:

**Define a loss function** `L(a, theta)` = cost of taking action `a` when the true state is `theta`.

**Compute expected posterior loss** for each action `a`:
```
EL(a) = E_{theta ~ posterior}[L(a, theta)]
```

**Take the action with minimum expected loss**.

Common loss functions:
- **Squared error** `(a - theta)^2`: optimal action is the posterior mean
- **Absolute error** `|a - theta|`: optimal action is the posterior median
- **0-1 loss** `1[a != theta]`: optimal action is the posterior mode
- **Asymmetric loss**: when false positives and false negatives have different costs

In [ ]:
# Asymmetric loss: medical test decision
# Action: treat or don't treat
# State: disease present or absent
# Costs:
#   Treat when no disease (false positive): cost = C_fp (side effects, expense)
#   Don't treat when disease (false negative): cost = C_fn (patient suffers, worse outcomes)

np.random.seed(42)

def optimal_threshold_asymmetric(posterior_p_disease, c_fp, c_fn):
    """Find the probability threshold that minimizes expected loss.
    
    Expected loss for treating: c_fp * P(no disease) = c_fp * (1 - p)
    Expected loss for not treating: c_fn * P(disease) = c_fn * p
    Treat when: c_fn * p > c_fp * (1 - p)
                   p > c_fp / (c_fp + c_fn)
    """
    threshold = c_fp / (c_fp + c_fn)
    decisions = (posterior_p_disease > threshold).astype(int)
    
    expected_loss = np.mean(
        np.where(decisions == 1,
                 c_fp * (1 - posterior_p_disease),  # treat: pay c_fp when no disease
                 c_fn * posterior_p_disease)          # don't treat: pay c_fn when disease
    )
    return threshold, decisions, expected_loss

# Patient with ambiguous test results
# Prior: 15% disease prevalence
# Posterior after positive test: 40-60% (uncertain)
patient_posterior_p = np.random.beta(4, 6, 10000)  # posterior centered ~0.4

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

loss_configs = [
    ('Symmetric\n(C_fp = C_fn = 1)', 1, 1),
    ('Conservative\n(C_fn = 5*C_fp)', 1, 5),
    ('Cost-sensitive\n(C_fp = 3*C_fn)', 3, 1),
]

for ax, (label, c_fp, c_fn) in zip(axes, loss_configs):
    threshold, decisions, exp_loss = optimal_threshold_asymmetric(
        patient_posterior_p, c_fp, c_fn
    )
    p_treat = decisions.mean()
    
    xr = np.linspace(0, 1, 300)
    kde = stats.gaussian_kde(patient_posterior_p)
    ax.fill_between(xr, kde(xr), alpha=0.4, color='steelblue')
    ax.plot(xr, kde(xr), 'steelblue', lw=1.5, label=f'P(disease) posterior')
    ax.axvline(threshold, color='red', lw=2, linestyle='--',
               label=f'Threshold: {threshold:.2f}')
    ax.axvline(patient_posterior_p.mean(), color='black', lw=1.5, linestyle=':',
               label=f'Post. mean: {patient_posterior_p.mean():.2f}')
    
    # Shade treatment region
    treat_x = xr[xr >= threshold]
    ax.fill_between(treat_x, kde(treat_x), alpha=0.15, color='red', label='Treat region')
    
    ax.set_xlabel('P(disease)')
    ax.set_title(f'{label}\nThreshold={threshold:.2f}  P(treat)={p_treat:.2f}')
    ax.legend(fontsize=7)

plt.suptitle('Asymmetric Loss Functions Shift the Decision Threshold\n'
             'Higher cost of missing disease -> lower treatment threshold', fontsize=12)
plt.tight_layout()
plt.show()

print('Decision thresholds:')
print('  Symmetric (C_fp = C_fn):  treat when P(disease) > 0.50')
print('  Conservative (C_fn = 5):  treat when P(disease) > 0.17  <- lower bar')
print('  Cost-sensitive (C_fp = 3): treat when P(disease) > 0.75  <- higher bar')
print('\nKey insight: the optimal threshold is c_fp / (c_fp + c_fn)')
print('Only when costs are equal does the threshold match 0.5')

## Part 4 — ROPE in Practice: A/B Test Analysis

Let's apply ROPE to a concrete A/B test scenario. We're testing a new checkout button color. The conversion rate difference needs to be at least 0.5% to be worth the engineering effort to deploy. Anything smaller is practically equivalent to the control.

In [ ]:
np.random.seed(42)

# A/B test data: two variants
# Control: 5000 visitors, 240 conversions
# Treatment: 5000 visitors, 265 conversions
n_control,   k_control   = 5000, 240
n_treatment, k_treatment = 5000, 265

# Bayesian analysis: Beta posteriors (conjugate)
alpha_prior, beta_prior = 1, 1

# Posterior distributions
post_control   = stats.beta(alpha_prior + k_control,   beta_prior + n_control   - k_control)
post_treatment = stats.beta(alpha_prior + k_treatment, beta_prior + n_treatment - k_treatment)

# Monte Carlo samples of the difference
n_samples = 100_000
control_samp   = post_control.rvs(n_samples)
treatment_samp = post_treatment.rvs(n_samples)
diff_samp      = treatment_samp - control_samp

# ROPE: differences < 0.5% are practically equivalent
rope_ab_lo, rope_ab_hi = -0.005, 0.005

result_ab = rope_decision(diff_samp, rope_ab_lo, rope_ab_hi)

print('A/B Test Results: Checkout Button Color')
print('=' * 50)
print(f'Control:   {k_control}/{n_control} = {k_control/n_control:.2%}')
print(f'Treatment: {k_treatment}/{n_treatment} = {k_treatment/n_treatment:.2%}')
print(f'ROPE: ({rope_ab_lo:.1%}, {rope_ab_hi:.1%}) -- differences smaller than this are negligible')
print()
print(f'Posterior mean difference: {diff_samp.mean():.4f} ({diff_samp.mean()*100:.2f}%)')
print(f'94% HDI: [{result_ab["hdi"][0]*100:.2f}%, {result_ab["hdi"][1]*100:.2f}%]')
print(f'P(treatment > control): {np.mean(diff_samp > 0):.3f}')
print(f'P(diff inside ROPE):    {result_ab["p_inside_rope"]:.3f}')
print(f'Decision: {result_ab["decision"]}')

In [ ]:
# Visualize the A/B test ROPE analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Marginal posteriors
ax = axes[0]
x_ctrl = np.linspace(0.03, 0.08, 300)
ax.fill_between(x_ctrl, post_control.pdf(x_ctrl), alpha=0.4, color='steelblue', label='Control')
ax.fill_between(x_ctrl, post_treatment.pdf(x_ctrl), alpha=0.4, color='darkorange', label='Treatment')
ax.plot(x_ctrl, post_control.pdf(x_ctrl), 'steelblue', lw=2)
ax.plot(x_ctrl, post_treatment.pdf(x_ctrl), 'darkorange', lw=2)
ax.axvline(k_control/n_control, color='steelblue', linestyle='--', lw=1)
ax.axvline(k_treatment/n_treatment, color='darkorange', linestyle='--', lw=1)
ax.set_xlabel('Conversion Rate'); ax.set_ylabel('Density')
ax.set_title('Posterior Distributions\nControl vs Treatment')
ax.legend(fontsize=9)

# Difference posterior with ROPE
ax = axes[1]
xr = np.linspace(diff_samp.min(), diff_samp.max(), 400)
kde = stats.gaussian_kde(diff_samp)
ax.fill_between(xr, kde(xr), alpha=0.35, color='steelblue')
ax.plot(xr, kde(xr), 'steelblue', lw=2)

# ROPE
ax.axvspan(rope_ab_lo, rope_ab_hi, alpha=0.2, color='gray', label=f'ROPE [{rope_ab_lo*100:.1f}%, {rope_ab_hi*100:.1f}%]')
ax.axvline(rope_ab_lo, color='gray', linestyle='--', lw=1.5)
ax.axvline(rope_ab_hi, color='gray', linestyle='--', lw=1.5)

# HDI
hdi_lo_ab, hdi_hi_ab = result_ab['hdi']
ax.axvline(hdi_lo_ab, color='red', lw=2.5, label=f'94% HDI: [{hdi_lo_ab*100:.2f}%, {hdi_hi_ab*100:.2f}%]')
ax.axvline(hdi_hi_ab, color='red', lw=2.5)

ax.axvline(0, color='black', linestyle=':', lw=1, alpha=0.7)
ax.set_xlabel('Treatment - Control (conversion rate)')
ax.set_ylabel('Density')
ax.set_title(f'Posterior Difference with ROPE\nDecision: {result_ab["decision"]}')
ax.legend(fontsize=8)

# Format x-axis as percentages
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.1f}%'))

plt.suptitle('ROPE Analysis: A/B Test Checkout Button', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# What if the true effect were different? Sample size sensitivity analysis
print('Sensitivity Analysis: Effect Size vs ROPE Decision')
print('=' * 65)
print(f'ROPE: +-0.5% | n=5000 per arm | Control rate=4.8%')
print()

true_effects = [0.001, 0.003, 0.005, 0.008, 0.015, 0.025]
print(f'{"True effect":15} {"P(> control)":14} {"P(inside ROPE)":16} {"Decision"}')
print('-' * 68)

for delta in true_effects:
    # Simulate data with this effect
    k_c = int(0.048 * n_control)
    k_t = int((0.048 + delta) * n_treatment)
    pc = stats.beta(1 + k_c, 1 + n_control - k_c).rvs(50000)
    pt = stats.beta(1 + k_t, 1 + n_treatment - k_t).rvs(50000)
    d  = pt - pc
    res = rope_decision(d, rope_ab_lo, rope_ab_hi)
    p_better = np.mean(d > 0)
    short_dec = res['decision'].split(' ')[0]
    print(f'{delta*100:>10.1f}%  {p_better:>13.3f}  {res["p_inside_rope"]:>15.3f}  {res["decision"]}')

## Part 5 — ROPE via ArviZ

ArviZ has built-in ROPE support through `az.plot_posterior()` with `rope` argument and `az.summary()` with `rope` parameter.

In [ ]:
# Bayesian model of the A/B test with ROPE visualization via ArviZ

with pm.Model() as ab_model:
    # Priors
    p_control   = pm.Beta('p_control',   alpha=1, beta=1)
    p_treatment = pm.Beta('p_treatment', alpha=1, beta=1)
    
    # Derived: rate difference
    delta = pm.Deterministic('delta', p_treatment - p_control)
    
    # Likelihoods
    obs_c = pm.Binomial('obs_c', n=n_control,   p=p_control,   observed=k_control)
    obs_t = pm.Binomial('obs_t', n=n_treatment, p=p_treatment, observed=k_treatment)
    
    trace_ab = pm.sample(draws=5000, tune=1000, chains=4,
                          random_seed=42, progressbar=False)

# ArviZ ROPE analysis
rope_dict = {'delta': [{'rope': (rope_ab_lo, rope_ab_hi)}]}

fig, ax = plt.subplots(figsize=(10, 4))
az.plot_posterior(
    trace_ab, var_names=['delta'],
    rope={'delta': [{'rope': (rope_ab_lo, rope_ab_hi)}]},
    hdi_prob=0.94,
    point_estimate='mean',
    ax=ax
)
ax.set_title('A/B Test: Posterior Difference with ROPE (ArviZ)\n'
             'Gray region = ROPE (differences < 0.5% are practically equivalent)')
plt.tight_layout()
plt.show()

# ArviZ summary with ROPE
print('ArviZ summary with ROPE:')
print(az.summary(trace_ab, var_names=['delta'],
                  rope={'delta': (rope_ab_lo, rope_ab_hi)}).to_string())

## Summary

**ROPE Decision Framework**:
1. Define ROPE `[lo, hi]` based on domain knowledge (smallest practically meaningful effect)
2. Compute 94% HDI of the posterior for the parameter of interest
3. Compare HDI to ROPE:
   - HDI fully inside ROPE -> accept null (practically equivalent)
   - HDI fully outside ROPE -> reject null (effect is meaningful)
   - HDI overlaps ROPE -> inconclusive (collect more data)

**ArviZ ROPE API**:
```python
# In az.plot_posterior:
az.plot_posterior(trace, var_names=['delta'],
                  rope={'delta': [{'rope': (-0.005, 0.005)}]},
                  hdi_prob=0.94)

# In az.summary:
az.summary(trace, rope={'delta': (-0.005, 0.005)})
```

**Loss function framework**:
- Optimal action = argmin over actions of E[L(action, theta) | data]
- Symmetric loss -> threshold at 0.5
- Asymmetric costs (e.g., medical: missing disease is worse than false alarm) -> threshold = C_fp / (C_fp + C_fn)

**HDI vs ETI**: use HDI for ROPE analysis — it finds the shortest interval covering the highest-density region. For skewed or multimodal posteriors, ETI and HDI differ significantly.

**Next**: Notebook 13 — Bayesian A/B Testing. Full end-to-end Bayesian test: from data collection to decision, with ROPE, stopping rules, and comparison to frequentist p-values.